# BlackJAX pixelized source reconstruction demo

This notebook simulates a lensed image from a Sersic source and fits it using a
Voronoi pixelization with a Delaunay-based mapping. The pixelized source values
are solved by linear inversion inside the likelihood.

Note: the Delaunay topology is fixed from an initial cache (host-side SciPy).
If lens parameters change enough to alter the topology, rebuild the cache.


In [1]:
# Keep JAX on CPU to avoid CUDA plugin warnings in minimal environments
import os
# os.environ.setdefault("JAX_PLATFORM_NAME", "cpu")

import sys
sys.path.insert(0, "..")

import jax
import jax.numpy as jnp
import jax.random as jr
import jax.nn as jnn
import numpy as np
import matplotlib.pyplot as plt

import blackjax
from blackjax.adaptation.window_adaptation import window_adaptation

from jax_lens.lens.tracer import (
    TracerConfig,
    PlaneConfig,
    traced_grid_list,
    tracer_image,
)
from jax_lens.fitting.imaging import convolve_image, log_likelihood
from jax_lens.pipeline import create_likelihood_fn
from jax_lens.pixelization import (
    PixelizationConfig,
    build_pixelization_cache_from_tracer,
    pixelized_source_reconstruction,
)

jax.config.update("jax_enable_x64", True)
plt.rcParams["figure.figsize"] = (12, 4)


In [2]:
# Utilities: grid, PSF, and parameter transforms

def create_grid(n_pixels: int, pixel_scale: float) -> jnp.ndarray:
    coords = jnp.linspace(-(n_pixels - 1) / 2 * pixel_scale, (n_pixels - 1) / 2 * pixel_scale, n_pixels)
    yy, xx = jnp.meshgrid(coords, coords, indexing="ij")
    return jnp.stack([yy, xx], axis=-1)

def create_psf(size: int = 13, sigma: float = 1.5) -> jnp.ndarray:
    half = (size - 1) / 2
    coords = jnp.linspace(-half, half, size)
    yy, xx = jnp.meshgrid(coords, coords, indexing="ij")
    psf = jnp.exp(-0.5 * (yy**2 + xx**2) / sigma**2)
    return psf / psf.sum()

def wrap_angle(phi: jnp.ndarray) -> jnp.ndarray:
    return (phi + jnp.pi) % (2 * jnp.pi) - jnp.pi

def softplus_pos(raw: jnp.ndarray, floor: float = 1e-3) -> jnp.ndarray:
    return floor + jnn.softplus(raw)

def inv_softplus_pos(val: jnp.ndarray, floor: float = 1e-3) -> jnp.ndarray:
    v = jnp.maximum(val - floor, 1e-8)
    return jnp.log(jnp.expm1(v))

def bounded_from_raw(raw: jnp.ndarray, low: float, high: float) -> jnp.ndarray:
    return low + (high - low) * jnn.sigmoid(raw)

def inv_bounded_from_raw(val: jnp.ndarray, low: float, high: float) -> jnp.ndarray:
    v = (val - low) / (high - low)
    v = jnp.clip(v, 1e-6, 1.0 - 1e-6)
    return jnp.log(v) - jnp.log1p(-v)


In [3]:
# Simulate mock data with a Sersic source

n_pixels = 80
pixel_scale = 0.05

grid = create_grid(n_pixels, pixel_scale)
grid_flat = grid.reshape(-1, 2)

true_params = {
    "lens_mass": {
        "centre": jnp.array([0.0, 0.0]),
        "einstein_radius": 1.05,
        "axis_ratio": 0.82,
        "angle": 0.5,
    },
    "source_light": {
        "centre": jnp.array([0.08, 0.04]),
        "intensity": 1.2,
        "effective_radius": 0.3,
        "sersic_index": 1.2,
        "axis_ratio": 0.7,
        "angle": 1.0,
    },
}

psf = create_psf(size=13, sigma=1.5)

true_image = tracer_image(
    grid=grid_flat,
    config=TracerConfig(planes=(
        PlaneConfig(redshift=0.5, light_profile_types=(), mass_profile_types=("sie",)),
        PlaneConfig(redshift=1.0, light_profile_types=("sersic",), mass_profile_types=()),
    )),
    params={
        "planes": [
            {"mass": [true_params["lens_mass"]], "light": []},
            {"mass": [], "light": [true_params["source_light"]]},
        ]
    },
)
true_image = true_image.reshape(n_pixels, n_pixels)
true_image = convolve_image(true_image, psf, padding="edge")

noise_sigma = 0.2
noise_map = jnp.ones_like(true_image) * noise_sigma
rng = np.random.default_rng(42)
noise = noise_sigma * rng.standard_normal(size=(n_pixels, n_pixels))
data = np.array(true_image) + noise

data_flat = jnp.asarray(data).reshape(-1)
noise_flat = noise_map.reshape(-1)


ERROR:2026-01-13 12:18:06,341:jax._src.xla_bridge:477: Jax plugin configuration error: Exception when calling jax_plugins.xla_cuda12.initialize()
Traceback (most recent call last):
  File "/home/nataliehogg/Documents/Codes/myenvs/EverythingDev/venv/lib/python3.12/site-packages/jax/_src/xla_bridge.py", line 475, in discover_pjrt_plugins
    plugin_module.initialize()
  File "/home/nataliehogg/Documents/Codes/myenvs/EverythingDev/venv/lib/python3.12/site-packages/jax_plugins/xla_cuda12/__init__.py", line 328, in initialize
    _check_cuda_versions(raise_on_first_error=True)
  File "/home/nataliehogg/Documents/Codes/myenvs/EverythingDev/venv/lib/python3.12/site-packages/jax_plugins/xla_cuda12/__init__.py", line 285, in _check_cuda_versions
    local_device_count = cuda_versions.cuda_device_count()
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: jaxlib/cuda/versions_helpers.cc:113: operation cuInit(0) failed: CUDA_ERROR_UNKNOWN


In [4]:
# Build pixelization cache (fixed topology) and likelihood

lens_plane = PlaneConfig(
    redshift=0.5,
    light_profile_types=(),
    mass_profile_types=("sie",),
)
source_plane = PlaneConfig(
    redshift=1.0,
    light_profile_types=(),
    mass_profile_types=(),
)
config = TracerConfig(planes=(lens_plane, source_plane))

params_cache = {
    "planes": [
        {"mass": [true_params["lens_mass"]], "light": []},
        {"mass": [], "light": []},
    ]
}

traced_true = traced_grid_list(grid_flat, config, params_cache)[-1]
t = np.array(traced_true)
ymin, ymax = t[:, 0].min(), t[:, 0].max()
xmin, xmax = t[:, 1].min(), t[:, 1].max()

n_side = 28  # ~784 seeds
margin = 0.05
ys = np.linspace(ymin + margin, ymax - margin, n_side)
xs = np.linspace(xmin + margin, xmax - margin, n_side)
yy, xx = np.meshgrid(ys, xs, indexing="ij")
seeds = np.stack([yy.ravel(), xx.ravel()], axis=-1)

cache = build_pixelization_cache_from_tracer(
    grid=grid_flat,
    config=config,
    params=params_cache,
    seeds=seeds,
)

pix_cfg = PixelizationConfig(
    regularization_weight=1.0,
    solver="dense",
    seeds=jnp.asarray(seeds),
)

log_like_fn = create_likelihood_fn(
    config=config,
    grid=grid_flat,
    data=data_flat,
    noise_map=noise_flat,
    psf=psf,
    image_shape=(n_pixels, n_pixels),
    source_mode="pixelization",
    pixelization=pix_cfg,
)
log_like_fn = jax.jit(log_like_fn)


In [5]:
# Parameterization: fit only lens mass parameters

def unpack_theta(theta: jnp.ndarray) -> dict:
    centre = theta[:2]
    einstein = softplus_pos(theta[2], floor=1e-3)
    axis_ratio = bounded_from_raw(theta[3], 0.2, 0.99)
    angle = wrap_angle(theta[4])

    return {
        "planes": [
            {
                "mass": [
                    {
                        "centre": centre,
                        "einstein_radius": einstein,
                        "axis_ratio": axis_ratio,
                        "angle": angle,
                    }
                ],
                "light": [],
            },
            {"mass": [], "light": []},
        ]
    }

theta_true = jnp.array([
    true_params["lens_mass"]["centre"][0],
    true_params["lens_mass"]["centre"][1],
    inv_softplus_pos(true_params["lens_mass"]["einstein_radius"], floor=1e-3),
    inv_bounded_from_raw(true_params["lens_mass"]["axis_ratio"], 0.2, 0.99),
    true_params["lens_mass"]["angle"],
])

prior_loc = theta_true
prior_scale = jnp.array([0.1, 0.1, 0.3, 0.6, 0.6])

@jax.jit
def log_prior(theta: jnp.ndarray) -> jnp.ndarray:
    z = (theta - prior_loc) / prior_scale
    return -0.5 * jnp.sum(z**2) - jnp.sum(jnp.log(prior_scale * jnp.sqrt(2 * jnp.pi)))

@jax.jit
def log_posterior(theta: jnp.ndarray) -> jnp.ndarray:
    params = unpack_theta(theta)
    return log_like_fn(params, cache) + log_prior(theta)

log_posterior(theta_true)


KeyboardInterrupt: 

In [ ]:
# Run NUTS with dual-averaging adaptation

rng = jr.PRNGKey(10)
rng, rng_init, rng_adapt, rng_sample = jr.split(rng, 4)

init_theta = theta_true + 0.05 * jr.normal(rng_init, theta_true.shape)

adapt = window_adaptation(
    blackjax.nuts,
    log_posterior,
    initial_step_size=0.02,
    target_acceptance_rate=0.8,
    is_mass_matrix_diagonal=True,
)
adapt_results, adapt_info = adapt.run(rng_adapt, init_theta, num_steps=400)

nuts = blackjax.nuts.differentiable(
    log_posterior,
    **adapt_results.parameters,
)

@jax.jit
def one_step(state, key):
    new_state, info = nuts.step(key, state)
    return new_state, (new_state, info)

num_samples = 400
keys = jr.split(rng_sample, num_samples)
state, (states, infos) = jax.lax.scan(one_step, adapt_results.state, keys)
samples = states.position


In [ ]:
# Posterior summary and reconstruction

burn_in = 100
posterior_tree = jax.vmap(unpack_theta)(samples[burn_in:])
mean_params = jax.tree.map(lambda x: jnp.mean(x, axis=0), posterior_tree)

print("Einstein radius (mean):", float(mean_params["planes"][0]["mass"][0]["einstein_radius"]))
print("Axis ratio (mean):", float(mean_params["planes"][0]["mass"][0]["axis_ratio"]))
print("Centre (mean y, x):", np.round(np.array(mean_params["planes"][0]["mass"][0]["centre"]), 3))

traced_mean = traced_grid_list(grid_flat, config, mean_params)[-1]
source_model, source_values = pixelized_source_reconstruction(
    points=traced_mean,
    seeds=jnp.asarray(seeds),
    cache=cache,
    data=data_flat,
    noise_map=noise_flat,
    regularization_weight=pix_cfg.regularization_weight,
    solver=pix_cfg.solver,
    psf=psf,
    image_shape=(n_pixels, n_pixels),
)

model_recon = source_model.reshape(n_pixels, n_pixels)

fig, axes = plt.subplots(2, 2, figsize=(10, 9))
axes = axes.ravel()

im0 = axes[0].imshow(true_image, origin="lower", cmap="magma")
axes[0].set_title("True Lensed Image (PSF)")
plt.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(data, origin="lower", cmap="magma")
axes[1].set_title("Noisy Data")
plt.colorbar(im1, ax=axes[1], fraction=0.046)

im2 = axes[2].imshow(model_recon, origin="lower", cmap="magma")
axes[2].set_title("Pixelized Reconstruction")
plt.colorbar(im2, ax=axes[2], fraction=0.046)

im3 = axes[3].imshow(data - np.array(model_recon), origin="lower", cmap="coolwarm")
axes[3].set_title("Residuals")
plt.colorbar(im3, ax=axes[3], fraction=0.046)

plt.tight_layout()
plt.show()

plt.figure(figsize=(5, 4))
plt.scatter(seeds[:, 1], seeds[:, 0], c=np.array(source_values), s=12, cmap="magma")
plt.gca().invert_yaxis()
plt.title("Recovered Source Values on Voronoi Seeds")
plt.colorbar()
plt.show()
